Three files have disease info:  

disease-gene.csv

disease-therapeuticarea.csv

drug-disease.csv

Uses ORPHA, HPO, and MONDO (or other) disease

ORPHA we can keep

HPO is not a disease, so make it a separate problem

# Radboud: map id to disease

for DOID, the path is:
DOID -> BIoPortal DOID Record (get label), follow mappings -> look for the SNOMEDCT mapping -> get ID

In [68]:
def efo_special_case(id)
    snomed = false
    url = "https://data.bioontology.org/ontologies/EFO/classes/http%3A%2F%2Fwww.ebi.ac.uk%2Fefo%2F#{id}?apikey=74027bd8-6be0-4329-be22-aa3717f97243"

    headers = {
    "Accept"     => "application/json",
    "User-Agent" => "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36"
    }

    begin
        response = RestClient.get(url, headers: headers)
    rescue
        return false
    end
    
    data = JSON.parse(response.body)

    if prefname = data.dig('prefLabel')
        step2 = "https://data.bioontology.org/search?q=#{prefname}&ontologies=SNOMEDCT&apikey=74027bd8-6be0-4329-be22-aa3717f97243"
#         warn "getting #{step2}"
        begin
            response = RestClient.get(step2, headers: headers)
        rescue
            return false
        end
        data = JSON.parse(response)
        records = data.dig('collection')
        records.each do |r|
#             warn r['prefLabel']
#             warn prefname
#             warn
            next unless r['prefLabel'].downcase == prefname.downcase
            snomed = r['@id']
            break
        end
    end
    if snomed
        return [id, snomed, prefname]
    else
        return false
    end
end
efo_special_case("EFO_0000713")

["EFO_0000713", "http://purl.bioontology.org/ontology/SNOMEDCT/21454007", "subarachnoid hemorrhage"]

In [59]:
require "rest-client"
require 'json'


def map_id_to_disease(name)
  return false unless (name =~ /EFO_/ || name =~ /DOID_(.*)/ || name =~ /MONDO_(.*)/ || name =~ /Orphanet_(.*)/)
  identifier = $1
  api_key = ENV["APIKEY"] # Replace with BioPortal API key
  begin
    if name =~ /DOID/
      url =  "https://data.bioontology.org/search?q=DOID:#{identifier}&ontologies=DOID&require_exact_match=true&apikey=#{api_key}"
    elsif name =~ /MONDO/
      url =  "https://data.bioontology.org/search?q=MONDO:#{identifier}&ontologies=MONDO&require_exact_match=true&apikey=#{api_key}"      
    else
      url =  "https://data.bioontology.org/search?q=Orphanet:#{identifier}&ontologies=ORDO&require_exact_match=true&apikey=74027bd8-6be0-4329-be22-aa3717f97243"
    end
#     warn url
    response = RestClient.get(url)
  rescue StandardError => e
    warn "No data found for #{name} #{e.inspect}\n"
    return false
  end
  mappings = []
  data = JSON.parse(response)

#  if hit = data.dig('collection', 0)
  if hit = data.dig('collection')
    hit.each do |h|
      disease_name = h&.dig('prefLabel')
      uri = h&.dig('@id')
      linksurl = h&.dig('links', 'mappings')
      mappings << { id: name, uri: uri, disease_name: disease_name, linksurl: linksurl}
    end
    return mappings
  else
    warn "No data found for #{name}\n"
    return false
  end

rescue StandardError => e
  warn "No data found for #{name} Error: #{e.inspect}\n"
  return false
end

:map_id_to_disease

In [13]:
# test
["DOID_7551", "MONDO_0003243", "Orphanet_101010"].each do |name|
  warn map_id_to_disease(name)
end
  

{:id=>"DOID_7551", :uri=>"http://purl.obolibrary.org/obo/DOID_7551", :disease_name=>"gonorrhea", :linksurl=>"https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_7551/mappings"}
{:id=>"MONDO_0003243", :uri=>"http://purl.obolibrary.org/obo/MONDO_0003243", :disease_name=>"hepatocellular clear cell carcinoma", :linksurl=>"https://data.bioontology.org/ontologies/MONDO/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FMONDO_0003243/mappings"}
{:id=>"Orphanet_101010", :uri=>"http://www.orpha.net/ORDO/Orphanet_101010", :disease_name=>"Autosomal spastic paraplegia type 30", :linksurl=>"https://data.bioontology.org/ontologies/ORDO/classes/http%3A%2F%2Fwww.orpha.net%2FORDO%2FOrphanet_101010/mappings"}


["DOID_7551", "MONDO_0003243", "Orphanet_101010"]

In [14]:
def get_snomed_from_mapping(mapping)
  if mapping =~ /MONDO/ 
    type = "MONDO"
  else
    type = "DOID_ORDO"  # mapping =~ /Orphanet/
  end

  api_key = ENV["APIKEY"] # Replace with BioPortal API key
  begin
    url = mapping+"?apikey=#{api_key}"
#     warn url
    response = RestClient.get(url)
  rescue StandardError => e
    warn "No data found for mapping #{e.inspect}\n"
    return false
  end
  mappings = []
  data = JSON.parse(response)

  if type == "DOID_ORDO"
    data.each do |mapp|
      mapp["classes"].each do |thisclass|
        next unless thisclass["@id"] =~ /SNOMEDCT/
        return {"snomed": thisclass["@id"]}
      end
    end
    warn "no SNOMED found for DOID/ORDO #{mapping}"
    return false
  else
    data.each do |mapp|
      mapp["classes"].each do |thisclass|
        next unless thisclass["@id"] =~ /(DOID_\d+)/  # if we start with MONDO, thre is no snomed,but there is a DOID
        doid = $1
        results = map_id_to_disease(doid) # go back to the beginning
        if results
          results = results.first # (can only be one doid)
          snomedres = get_snomed_from_mapping(results[:linksurl])  # recursive, but this time its a DOID
          if snomedres
            return snomedres  # {"snomed": thisclass["@id"]} from the routine above
          else
            warn "found no snomed for #{mapping}"
            return false
          end
        else
          warn "found no results for xreffed doid #{doid}"
          return false
        end
      end
    end
  end
    
  warn "No SNOMED found in #{mapping}\n"
  return false
rescue StandardError => e
  warn  "parsing or other error in #{mapping} Error: #{e.inspect}\n"
  return false
end

:get_snomed_from_mapping

In [15]:
# test
["DOID_7551", "MONDO_0003243", "Orphanet_101010"].each do |name|
  results = map_id_to_disease(name)
  if results
    results = results.first # (can only be one)
    nextres = get_snomed_from_mapping(results[:linksurl])
    if nextres
      results.merge! nextres
    end
    puts results
  else
    warn "found no results for #{name}"
  end
end


{:id=>"DOID_7551", :uri=>"http://purl.obolibrary.org/obo/DOID_7551", :disease_name=>"gonorrhea", :linksurl=>"https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_7551/mappings", :snomed=>"http://purl.bioontology.org/ontology/SNOMEDCT/15628003"}


no SNOMED found for DOID/ORDO https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_5016/mappings
found no snomed for https://data.bioontology.org/ontologies/MONDO/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FMONDO_0003243/mappings


{:id=>"MONDO_0003243", :uri=>"http://purl.obolibrary.org/obo/MONDO_0003243", :disease_name=>"hepatocellular clear cell carcinoma", :linksurl=>"https://data.bioontology.org/ontologies/MONDO/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FMONDO_0003243/mappings"}
{:id=>"Orphanet_101010", :uri=>"http://www.orpha.net/ORDO/Orphanet_101010", :disease_name=>"Autosomal spastic paraplegia type 30", :linksurl=>"https://data.bioontology.org/ontologies/ORDO/classes/http%3A%2F%2Fwww.orpha.net%2FORDO%2FOrphanet_101010/mappings", :snomed=>"http://purl.bioontology.org/ontology/SNOMEDCT/763377006"}


["DOID_7551", "MONDO_0003243", "Orphanet_101010"]

In [16]:
require 'csv'

filelist = ["disease-gene.csv","disease-therapeuticarea.csv"]  # in these two it is called "source"

diseaselist = {}
filelist.each do |csvfile|
  CSV.foreach("./rawdata/#{csvfile}", headers: true) do |row|
    next if row.size < 3

    source = row['source'].strip
#     next if source =~ /HP_/
    abort unless source
    diseaselist[source] = 1
  end
end

filelist = ["drug-disease.csv"]  # in these two it is called "target"

filelist.each do |csvfile|
  CSV.foreach("./rawdata/#{csvfile}", headers: true) do |row|
    next if row.size < 3

    source = row['target'].strip
#     next if source =~ /HP_/
    abort unless source
    diseaselist[source] = 1
  end
end

puts diseaselist.length  

9339


In [69]:
# write the mapping file
f = File.open("./mappings/diseases.map", "w")
e = File.open("./mappings/diseases-errors.txt", "w")

f.write CSV.generate_line(["source","snomedct","orpha","prefname"])

# spoeial case for EFO
diseaselist.each_key do |name|
    next unless name =~ /EFO_/
    id, snomed, prefname = efo_special_case(name)
    unless id
        warn "found no snomed for links in #{name}"
        e.write "found no snomed for links in #{name}\n"
        next
    else
        orpha = "https://fake.orphanet/not-found"
        warn snomed
        f.write CSV.generate_line([id,snomed,orpha,prefname])
    end
end
        

diseaselist.each_key do |name|
  warn name
  next if name =~ /HP_/  # not HPO terms
  next if name =~ /EFO_/  # not EFO terms
  next if name =~ /OTAR_/  # not EFO terms
  
  results = map_id_to_disease(name)
  if results
    results = results.first # (can only be one)
    nextres = get_snomed_from_mapping(results[:linksurl])
    if nextres
      results.merge! nextres
    else
      warn "found no snomed for links in #{name}"
      e.write "found no snomed for links in #{name}\n"
    end
    snomed = results[:snomed]
    snomed = "https://fake.snomed/not-found" unless snomed
    if results[:uri] =~ /orpha/
      orpha = results[:uri]
    else
      orpha = "https://fake.orphanet/not-found"
    end
    f.write CSV.generate_line([results[:id],snomed,orpha,results[:disease_name]])
    # {:id=>"Orphanet_101010", :uri=>"http://www.orpha.net/ORDO/Orphanet_101010", :disease_name=>"Autosomal spastic paraplegia type 30", :linksurl=>"https://data.bioontology.org/ontologies/ORDO/classes/http%3A%2F%2Fwww.orpha.net%2FORDO%2FOrphanet_101010/mappings", :snomed=>"http://purl.bioontology.org/ontology/SNOMEDCT/763377006"}
  else
    warn "found no results for #{name}"
    e.write "found no results for #{name}\n"
  end
end

http://purl.bioontology.org/ontology/SNOMEDCT/21454007
found no snomed for links in EFO_0001675
found no snomed for links in EFO_0000174
found no snomed for links in EFO_0003073
found no snomed for links in EFO_0003075
found no snomed for links in EFO_0000595
http://purl.bioontology.org/ontology/SNOMEDCT/26843008
found no snomed for links in EFO_0000233
found no snomed for links in EFO_0003027
http://purl.bioontology.org/ontology/SNOMEDCT/233964008
http://purl.bioontology.org/ontology/SNOMEDCT/51608009
found no snomed for links in EFO_0003046
http://purl.bioontology.org/ontology/SNOMEDCT/39553005
found no snomed for links in EFO_0003764
http://purl.bioontology.org/ontology/SNOMEDCT/116381000119105
found no snomed for links in EFO_0001357
http://purl.bioontology.org/ontology/SNOMEDCT/190817009
found no snomed for links in EFO_0002623
found no snomed for links in EFO_0003779
found no snomed for links in EFO_0000195
found no snomed for links in EFO_0003812
found no snomed for links in EFO

NoMethodError: undefined method `[]' for nil:NilClass

In [1]:
puts `pwd`



/home/osboxes/CODE/SIMPATHIC2/SKG_Mapping/radboud
